In [2]:
# وارد کردن کتابخانه‌ها
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split as test
from sklearn.pipeline import Pipeline as pl
from sklearn.compose import ColumnTransformer as ct
from sklearn.impute import SimpleImputer as si
from sklearn.preprocessing import OneHotEncoder as ohe
from sklearn.preprocessing import StandardScaler as std

# ثابت کردن Random Seed برای قابل تکرار بودن نتایج
RANDOM_STATE = 42

# خواندن داده 
df = pd.read_csv("../data/bank-full.csv", sep=";")

# نمایش تعداد سطرها و ستون‌ها
print(f"Number os Records: {df.shape[0]:,}")
print(f"Number os Columns: {df.shape[1]:,}")

Number os Records: 45,211
Number os Columns: 17


In [3]:
# نام ستون هدف
TARGET = "y"

# جدا کردن Featureها از Target
X = df.drop(columns=TARGET)

# ذخیره
y = df[TARGET]

# حذف duration به دلیل اینکه در سناریوی Prediction قبل از تماس
# در زمان پیش‌بینی در دسترس نیست
X = X.drop(columns=["duration"])

# بررسی ابعاد نهایی
print("Scale of Features:", X.shape)
print("Scale of Target:", y.shape)

Scale of Features: (45211, 15)
Scale of Target: (45211,)


In [4]:
# تقسیم داده به Train و بخش موقت
# 70 درصد برای Train و 30 درصد برای Validation + Test

X_train, X_temp, y_train, y_temp = test(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

In [5]:
# تقسیم 30 درصد باقی‌مانده به دو قسمت مساوی:
# 15 درصد Validation
# 15 درصد Test

X_val, X_test, y_val, y_test = test(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

# نمایش ابعاد
print("Train:")
print(X_train.shape)

print("\nValidation:")
print(X_val.shape)

print("\nTest:")
print(X_test.shape)

Train:
(31647, 15)

Validation:
(6782, 15)

Test:
(6782, 15)


In [6]:
# تابعی برای نمایش توزیع در هر بخش داده

def show_target_distribution(name, target):
    print(f"\n{name}")

    distribution = pd.DataFrame({
        "Count": target.value_counts(),
        "Percent": (target.value_counts(normalize=True) * 100).round(2)
    })

    print(distribution)


# بررسی توزیع Target در کل Dataset
show_target_distribution("Dataset", y)

# بررسی توزیع Target در Train
show_target_distribution("Train", y_train)

# بررسی توزیع Target در Validation
show_target_distribution("Validation", y_val)

# بررسی توزیع Target در Test
show_target_distribution("Test", y_test)


Dataset
     Count  Percent
y                  
no   39922     88.3
yes   5289     11.7

Train
     Count  Percent
y                  
no   27945     88.3
yes   3702     11.7

Validation
     Count  Percent
y                  
no    5988    88.29
yes    794    11.71

Test
     Count  Percent
y                  
no    5989    88.31
yes    793    11.69


In [7]:
# شناسایی Featureهای عددی
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# شناسایی Featureهای categorical
categorical_features = X_train.select_dtypes(
    include="object"
).columns.tolist()

print("Numeric Features:")
print(numeric_features)

print("\ncategorical Features:")
print(categorical_features)

Numeric Features:
['age', 'balance', 'day', 'campaign', 'pdays', 'previous']

categorical Features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


C:\Users\hp\AppData\Local\Temp\ipykernel_8848\1660650269.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [8]:
# Pipeline مربوط به Featureهای عددی

numeric_pipeline = pl(steps=[
    
    # اگر مقدار Missing وجود داشته باشد،
    # مقدار Median همان ستون جایگزین می‌شود
    ("imputer", si(strategy="median")),
    
    # استانداردسازی Featureهای عددی
    ("scaler", std())
])

In [9]:
# Pipeline مربوط به Featureهای categorical

categorical_pipeline = pl(steps=[
    
    # جایگزینی Missing Value با پرتکرارترین مقدار
    ("imputer", si(strategy="most_frequent")),
    
    # تبدیل متغیرهای categorical به Featureهای عددی
    ("encoder", ohe(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

In [10]:
# اعمال Pipeline عددی روی Featureهای عددی
# و Pipeline categorical روی Featureهای categorical

preprocessor = ct(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

In [11]:
# Pipeline نهایی preprocessing

preprocessing_pipeline = pl(steps=[
    ("preprocessor", preprocessor)
])

In [12]:
# یادگیری پارامترهای preprocessing فقط از Train
preprocessing_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['age','job','marital',...,'pdays','previous','poutcome']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated wi

In [13]:
# تبدیل داده‌های Train
X_train_processed = preprocessing_pipeline.transform(X_train)

# تبدیل Validation
X_val_processed = preprocessing_pipeline.transform(X_val)

# تبدیل Test
X_test_processed = preprocessing_pipeline.transform(X_test)

# نمایش ابعاد داده‌های تبدیل‌شده
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Train: (31647, 50)
Validation: (6782, 50)
Test: (6782, 50)


In [14]:
# بررسی اینکه آیا بعد از preprocessing مقدار Missing داریم یا خیر

print(
    "Number of Missing in Train:",
    np.isnan(X_train_processed).sum()
)

print(
    "Number of Missing in Validation:",
    np.isnan(X_val_processed).sum()
)

print(
    "Number of Missing in Test:",
    np.isnan(X_test_processed).sum()
)

Number of Missing in Train: 0
Number of Missing in Validation: 0
Number of Missing in Test: 0
